# 11 · Gradient boosting

[`07_trees`](07_trees.ipynb) builds trees independently (one deep tree, or a
bagged forest). **Gradient boosting** grows trees *sequentially*: each new
tree predicts the **residual error** of the ensemble so far, and its (shrunk)
prediction is added on top. Steps: start from a constant baseline, fit a
shallow tree to the residuals, add a fraction of its prediction to the running
total, recompute residuals, repeat.

Chaining weak learners this way typically beats one strong tree or a bagged
forest, at the cost of sequential (non-parallel) training and more sensitivity
to overfitting with too many rounds.

`GradientBoostingTree` ([`gradient_boosting.py`](../src/dolcestat/trees/gradient_boosting.py))
implements both regression and classification:

| | `impurity_measure` | baseline | residual fit at each round |
|---|---|---|---|
| **regression** | `"mse"` | training mean | $y - \hat{y}$ |
| **classification** | `"gini"` / `"entropy"` | per-class log-frequency | $\mathbb{1}[y = k] - \hat{p}_k$ |

Two details worth flagging:

- **Sub-trees always split on MSE**, regardless of `impurity_measure`.
  Boosting fits each tree to *continuous residuals*; a gini/entropy leaf in
  [`decision_tree.py`](../src/dolcestat/trees/decision_tree.py) predicts
  `np.bincount(y).argmax()`, which needs non-negative integer labels.
  `impurity_measure` selects the **task**, not the sub-tree's split criterion.
- **Classification is multinomial**: one forest per class boosts that class's
  log-odds, with a shared row-wise softmax converting the
  `(n_samples, n_classes)` score matrix to probabilities. Binary is just
  $K = 2$ of the same code — no separate binary path.

## A dataset where boosting should earn its keep

The target needs structure a **linear model** can't reach, but a **flexible
non-linear ensemble** can. Built from:

- **pairwise interaction** ($x_1 \times x_2$) — invisible to additive linear terms;
- **threshold/XOR effect** ($x_3, x_4$ same sign vs. opposite) — no single
  axis-aligned split separates it, but chained splits can;
- **smooth non-linearity** ($\sin$ of $x_5$) — approximable by shallow trees;
- five **pure noise** features, so overfitting has somewhere to hide.

In [1]:
import numpy as np
import polars as pl

from dolcestat.linear_models import LinearRegression
from dolcestat.neural_networks import (
    LU,
    DenseLayer,
    GradientDescent,
    MeanSquaredError,
    MiniBatchIteratingSampler,
    ReLU,
    Sequential,
)
from dolcestat.preprocessing import DolceSet
from dolcestat.trees import DecisionTree, GradientBoostingTree, RandomForest

rng = np.random.default_rng(0)
n = 800

x1 = rng.normal(0, 1, n)
x2 = rng.normal(0, 1, n)
x3 = rng.normal(0, 1, n)
x4 = rng.normal(0, 1, n)
x5 = rng.uniform(-3, 3, n)
noise = rng.normal(0, 1, (n, 5))

interaction = 2.5 * x1 * x2
xor_effect = 3.0 * np.where(np.sign(x3) == np.sign(x4), 1.0, -1.0)
smooth = 2.0 * np.sin(1.5 * x5)
y = interaction + xor_effect + smooth + rng.normal(0, 0.5, n)

cols = {"x1": x1, "x2": x2, "x3": x3, "x4": x4, "x5": x5}
cols.update({f"noise{i + 1}": noise[:, i] for i in range(5)})
cols["y"] = y
df = pl.DataFrame(cols)

order = rng.permutation(n)
train_rows, test_rows = order[:600].tolist(), order[600:].tolist()
train = DolceSet()
train.load_from_polars_dataframe(df[train_rows], target_col="y")
test = DolceSet()
test.load_from_polars_dataframe(df[test_rows], target_col="y")

print(f"train: {train.X.shape}, test: {test.X.shape}")
print(f"y range: [{y.min():.2f}, {y.max():.2f}], std: {y.std():.2f}")

train: (600, 10), test: (200, 10)
y range: [-16.00, 13.75], std: 4.19


## Baselines: linear regression, one tree, a forest, a network

`LinearRegression` only sees additive effects, so the interaction and XOR
terms show up as residual error. An unconstrained `DecisionTree` can represent
this structure, but with no depth limit it also memorises the five noise
features. `RandomForest` (bagging over random feature subsets) and a small
feed-forward network (`ReLU` layers, as in [`09_FFNN`](09_FFNN.ipynb)) trade
flexibility for lower variance.

These four fits take a couple of minutes — the forest grows 200 unconstrained
trees in pure NumPy.

In [2]:
linreg = LinearRegression(optimizer="closed form").fit(train)
lr_rmse = linreg.predict(test).rmse()

tree = DecisionTree(max_depth=None, min_samples_to_split=5, impurity_measure="mse").fit(train)
tree_train_rmse = tree.predict(train).rmse()
tree_rmse = tree.predict(test).rmse()

np.random.seed(0)  # forests draw from NumPy's global random generator
forest = RandomForest(
    n_trees=200, n_feat=5, max_depth=None, min_samples_to_split=10,
    impurity_measure="mse",
).fit(train)
forest_rmse = forest.predict(test).rmse()

np.random.seed(0)
nn = Sequential(
    DenseLayer(ReLU(), train.X.shape[1], 32),
    DenseLayer(ReLU(), 32, 16),
    DenseLayer(LU(), 16, 1),
)
nn.fit(
    train,
    loss=MeanSquaredError(),
    n_epochs=400,
    optimizer=GradientDescent(alpha=0.01),
    sampler=MiniBatchIteratingSampler(batch_size=32),
    tol=None,
)
nn_rmse = nn.predict(test).rmse()

print(f"{'model':<22s} {'train RMSE':>10s} {'test RMSE':>10s}")
print(f"{'linear regression':<22s} {'':>10s} {lr_rmse:>10.3f}")
print(f"{'single tree (mse)':<22s} {tree_train_rmse:>10.3f} {tree_rmse:>10.3f}")
print(f"{'random forest':<22s} {'':>10s} {forest_rmse:>10.3f}")
print(f"{'feed-forward NN':<22s} {'':>10s} {nn_rmse:>10.3f}")

model                  train RMSE  test RMSE
linear regression                      3.892
single tree (mse)           0.969      4.930
random forest                          3.247
feed-forward NN                        2.914


The lone tree overfits clearly: training RMSE ~0.97 vs. test RMSE ~4.93,
worse than linear regression. It splits on the noise columns too, and those
splits don't generalize. Linear regression lands at ~3.89 for the opposite
reason — too little flexibility to fit the interaction/XOR terms. Its test
RMSE is roughly the standard deviation of $y$: it explains almost nothing.

The random forest (~3.25) and small NN (~2.91) both improve by trading
per-model flexibility for variance reduction — bagging over feature subsets
for the forest, gradient descent over stacked non-linear layers for the NN.

## Boosting the same data

Same `data → fit → predict` shape as every other `dolcestat.trees` estimator.
Key hyperparameters:

- `max_forest_size` — number of **boosting rounds** (trees fit in sequence);
- `alpha` — **learning rate**, shrinking each tree's contribution before
  adding it to the running prediction;
- `max_depth` — kept deliberately **small**. Each tree only corrects the
  previous ensemble's residuals, so shallow high-bias trees are the point;
  capacity comes from the number of rounds, not tree size.

The defaults (`alpha=0.1`, `max_depth=3`, 100 rounds) follow the same reasoning.

`alpha` and `max_forest_size` trade off: a smaller learning rate needs more
rounds for the same fit, but usually generalises better.

In [3]:
gb = GradientBoostingTree(
    alpha=0.1,
    max_depth=3,
    min_samples_to_split=5,
    max_forest_size=200,
    impurity_measure="mse",
).fit(train)

gb_train_rmse = gb.predict(train).rmse()
gb_rmse = gb.predict(test).rmse()

print(f"{'model':<22s} {'train RMSE':>10s} {'test RMSE':>10s}")
print(f"{'linear regression':<22s} {'':>10s} {lr_rmse:>10.3f}")
print(f"{'single tree (mse)':<22s} {tree_train_rmse:>10.3f} {tree_rmse:>10.3f}")
print(f"{'random forest':<22s} {'':>10s} {forest_rmse:>10.3f}")
print(f"{'feed-forward NN':<22s} {'':>10s} {nn_rmse:>10.3f}")
print(f"{'gradient boosting':<22s} {gb_train_rmse:>10.3f} {gb_rmse:>10.3f}")
print(f"\ntrees in the ensemble: {len(gb.forest)}")
print(f"baseline (training mean): {gb.init_value:.3f}")

model                  train RMSE  test RMSE
linear regression                      3.892
single tree (mse)           0.969      4.930
random forest                          3.247
feed-forward NN                        2.914
gradient boosting           0.855      2.117

trees in the ensemble: 200
baseline (training mean): 0.176


Boosting wins: test RMSE ~2.12, vs. 2.91 (network) and 3.25 (forest). The
telling comparison is against the single tree — same CART splits, but the
boosted ensemble reaches *lower* training error (~0.86 vs ~0.97) while more
than halving test error. Depth-3 trees can't memorise noise individually, and
shrinking each by `alpha = 0.1` keeps any single tree from moving the
prediction far — the ensemble commits only to patterns that keep showing up
in residuals round after round.

This is the bias/variance story from the other side: the forest averages
*deep, independent, high-variance* trees to cancel variance; boosting *sums*
*shallow, sequential, high-bias* trees to chip away at bias. On a target
needing *coordinated* trees rather than merely numerous ones (interaction +
XOR + sine), the sequential approach wins.

## Classification

`impurity_measure="gini"` (or `"entropy"`) switches to the classification
objective: baseline becomes a per-class log-frequency vector, and each round
fits one tree per class against $\mathbb{1}[y = k] - \hat{p}_k$, with
$\hat{p}$ from the shared softmax over class scores. `predict()` softmaxes
accumulated scores into probabilities and hands them to an analyzer (binary:
`ClassificationAnalyzer`; three or more classes: `MulticlassAnalyzer`), so the
metrics from [`06_metrics`](06_metrics.ipynb) apply.

The target below is non-linear ($a_1 a_2$ interaction plus a sine), so
logistic regression would struggle.

In [4]:
rng_c = np.random.default_rng(1)
n_c = 600

a1 = rng_c.normal(0, 1, n_c)
a2 = rng_c.normal(0, 1, n_c)
logit = 2.0 * a1 * a2 + 1.5 * np.sin(2 * a1)
prob = 1 / (1 + np.exp(-logit))
label = (rng_c.uniform(0, 1, n_c) < prob).astype(int)

cdf = pl.DataFrame({"a1": a1, "a2": a2, "label": label})
c_order = rng_c.permutation(n_c)
c_train_rows, c_test_rows = c_order[:450].tolist(), c_order[450:].tolist()
c_train = DolceSet()
c_train.load_from_polars_dataframe(cdf[c_train_rows], target_col="label")
c_test = DolceSet()
c_test.load_from_polars_dataframe(cdf[c_test_rows], target_col="label")

gbc = GradientBoostingTree(
    alpha=0.1,
    max_depth=3,
    min_samples_to_split=5,
    max_forest_size=60,
    impurity_measure="gini",
).fit(c_train)

result = gbc.predict(c_test)
print(f"classes found:        {gbc.classes_}")
print(f"forests (one/class):  {len(gbc.forest)}")
print(f"trees per forest:     {len(gbc.forest[0])}")
print(f"log-odds baseline:    {np.round(gbc.init_value, 3)}")
print()
print(f"test accuracy:  {result.accuracy(0.5):.3f}")
print(f"test AUC:       {result.auc():.3f}")
print(f"test BCE:       {result.bce():.3f}")

classes found:        [0 1]
forests (one/class):  2
trees per forest:     60
log-odds baseline:    [-0.786 -0.608]

test accuracy:  0.713
test AUC:       0.792
test BCE:       0.584


### Three or more classes

Nothing changes in the fit: one forest per class, boosted over the shared softmax. `predict()` returns a `MulticlassAnalyzer` — confusion matrix, macro/weighted precision, recall and F1, log-loss and one-vs-rest AUC, as in [`06_metrics`](06_metrics.ipynb). Class labels must be `0, 1, …, K-1`.

In [5]:
rng_m = np.random.default_rng(2)
centres = np.array([[0.0, 0.0], [3.0, 3.0], [-3.0, 3.0]])
m_X = np.vstack([c + rng_m.normal(0, 1.0, (80, 2)) for c in centres])
m_y = np.repeat([0, 1, 2], 80)
m_order = rng_m.permutation(len(m_y))
m_train = DolceSet.from_arrays(m_X[m_order[:180]], m_y[m_order[:180]])
m_test = DolceSet.from_arrays(m_X[m_order[180:]], m_y[m_order[180:]])

gbm = GradientBoostingTree(
    alpha=0.3, max_depth=2, max_forest_size=30, impurity_measure="gini",
).fit(m_train)
multi = gbm.predict(m_test)

print(f"classes {gbm.classes_}: {len(gbm.forest)} forests of {len(gbm.forest[0])} trees")
print(type(multi).__name__)
print("confusion matrix (rows = true class, columns = predicted):")
print(multi.confusion_matrix())
print(f"test accuracy {multi.accuracy():.3f}   log-loss {multi.log_loss():.3f}   one-vs-rest AUC {multi.auc():.3f}")

classes [0 1 2]: 3 forests of 30 trees
MulticlassAnalyzer
confusion matrix (rows = true class, columns = predicted):
[[18  0  1]
 [ 0 21  0]
 [ 1  0 19]]
test accuracy 0.967   log-loss 0.165   one-vs-rest AUC 0.999


## Takeaways

- Boosting **sums** shallow, sequential, high-bias trees (each fit to what the
  ensemble still gets wrong); bagging **averages** deep, independent,
  high-variance ones. Both end up as many trees, for opposite reasons.
- `alpha` and `max_forest_size` are the real regularisers, alongside a small
  `max_depth`. Shrinking harder means needing more rounds — and usually
  generalising better.
- `impurity_measure` picks the **task** (`"mse"` → regression, `"gini"` /
  `"entropy"` → classification), while the sub-trees always split on MSE
  because they are fit to continuous residuals.
- Classification is multinomial throughout — one forest per class over a shared
  softmax — so binary is just the $K = 2$ case. With three or more classes
  `predict()` returns a `MulticlassAnalyzer`.